## Instance

Each openDAQ application starts with an _instance_. The instance is an entry-point object that sets up a localized environment of openDAQ objects (required for eg. logging, scheduling, loading libraries...). It also creates the root node of the openDAQ component tree - a virtual device. openDAQ provides a set of components that represent different data acquisition concepts - signals, devices, function blocks... and organizes them into a tree structure, starting with our Instance-created Device as its root node.

In the below code snippet, we create an Instance, in turn creating our root device. All calls to the instance will be forwarded to the root device itself.

In [ ]:

# This will require the openDAQ SDK to be installed. 
# If it is not installed, you can install it using pip from command line:
# pip install -q "opendaq>=3.40.0"
# or directly in a Jupyter notebook:
# %pip install -q "opendaq>=3.40.0"

import opendaq as daq
import os
import time

try:
    import opendaq as daq
except ImportError:
    raise ImportError("openDAQ SDK is not installed. Please install it from openDAQ GitHub:\n")        

# Check the openDAQ version: this tutorial was designed with version 3.40.0
if daq.__version__ < "3.40.0":
    raise RuntimeError("openDAQ version 3.40.0 or higher is required.")


### Telling the instance where to look for custom function blocks
Before creating the _instance_, let us tell openDAQ where to look for additional function block modules (e.g. your own custom build output).
`add_module_path` is added on top of the default module search locations (it doesn't replace them),
so the standard reference modules are still discovered.

In [ ]:
instance_builder = daq.InstanceBuilder()

appdata_path = os.environ.get('APPDATA')
if appdata_path:
	instance_builder.add_module_path(os.path.join(appdata_path, 'advantage-connect', 'custom_modules'))

daq_instance = instance_builder.build()

## Available devices

Each device can have a set of different sub-components as children. Amongst those, a device can have other sub-devices. When connecting to a device, we connect to/add the said device, and attach it as a child node of our main device (Instance). By using the `available_devices` device property, we can scan our system and network for openDAQ-enabled devices:

In [ ]:
available_devices = daq_instance.available_devices
for device in available_devices:
    print(device.name + ';', device.connection_string)

## Adding FusionLN

If Fusion LN module is connected to the local network, it should be listed in the _available devices_ list. Each device has a _connection string_. In the code below, put the connection string inside the brackets.


A FusionLN can also be added using only a static IP address, and we could disable the mDNS messages. However, this would prevent the use of the connection string `daq://HBK_200021` shown for `HBK3070-20002`, which is a smart connection string. A smart connection string means that during discovery, a device information object is created where information such as protocols used is stored.

Currently, FusionLN uses OPC-UA for configuration and a web socket connection for streaming. If you want to connect only to the streaming, please use `daq.lt://<IP>` (the IP address is shown on the display of the device module). If you also want to be able to configure the device, please use `daq.opcua://<IP>`.

In [ ]:
# Replace the connection string below with the one for your FusionLN device. You can find it in the output of the cell above.
my_fusionln_connection_string = 'daq://HBK_10003'
fusionln_device = daq_instance.add_device(my_fusionln_connection_string)

### Let's set the DAQ:

I have the following sensor configuration:

Channel 1: Mic B&K Type 4957, CCLD, TEDS. Requires 22.4 Hz HPF

Channel 2: Acc B&K Type 4507, CCLD, no TEDS. HPF 0.7 Hz

Channel 3: Demo motor B&K Type WB 1470, direct input, no TEDS,  HPF set to DC

Channel 4: Acc B&K Type 4508 B 004, CCLD, TEDS. HPF 0.7 Hz 

1. Let's set CCLD (= IEPE) on the channels which need it

In [ ]:
# Each channel has a "SignalConditioner" function block whose SensorType property
# selects the sensor mode (3 = Iepe, 6 = Voltage).
iepe_channels = [0,1,3] # Channels that require IEPE
voltage_channels = [2]  # Channels that require Voltage

for channel_index in range(4):
    channel: daq.IChannel  = fusionln_device.channels[channel_index]    # Use optional type annotation for better code completion in IDEs
    if channel_index in iepe_channels:
        sensor_type, sensor_name = 3, 'Iepe'  # 3 = Iepe
    else:
        sensor_type, sensor_name = 6, 'Voltage'  # 6 = Voltage

    fbSignalConditioner: daq.IFunctionBlock = channel.get_function_blocks(daq.LocalIdSearchFilter('SignalConditioner'))[0]
    fbSignalConditioner.set_property_value('SensorType', sensor_type)
    print(channel.global_id, '->', sensor_name)
    
    # 👇 Less efficient and more verbose...
    #for fb in channel.function_blocks:
    #    if fb.name == 'SignalConditioner':
    #        print('fb name ->', fb.local_id, fb.name)
    #        fb.set_property_value('SensorType', sensor_type)
    #        print(channel.global_id, '->', sensor_name)

# So how do you figure out what the sensor type numbers are?
# 1.) Using a visual tool like the OpenDAQ python app to inspect the function block properties.
# 2.) Using the SDK to query the property selection values:
if fbSignalConditioner:
    propSensorType = fbSignalConditioner.get_property('SensorType')
    sensorTypes: daq.IDict = propSensorType.selection_values
    for key, value in sensorTypes.items():
        print(key, value)


2. Set up the sampling rate.

Now let's set the sampling rate. When powered on or rebooted, the Fusion LN module is set to the highest possible sampling rate. Below we will change it to 4096 Sample per second.

Note: I set ALL channels to the lower sampling rate: I noticed that the network performance somehow drops in case I keep at least one channel at very high sampling rate. Maybe this channel must be disabled? 

In [ ]:
SAMPLE_RATE = 4096

# The "SampleRate" property lives on each channel's "Value" signal,
# e.g. global ID ".../IO/Connector1/InputChannels/Channel1/Sig/Value".
for channel in fusionln_device.channels[:4]:
    for signal in channel.signals:
        if signal.global_id.endswith('/Sig/Value'):
            signal.set_property_value('SampleRate', SAMPLE_RATE)
            print(signal.global_id, '-> SampleRate =', signal.get_property_value('SampleRate'))


3. Let's make the channels active

In [ ]:
# Check the status of all channels.
print("Initial channel status:")
for channel in fusionln_device.channels:
    print(channel.global_id, '-> active =', "yes" if channel.active else "no")

# Activate the first 4 channels.
for channel in fusionln_device.channels[:4]:
    channel.active = True

# Deactivate the remaining 2 channels.
for channel in fusionln_device.channels[4:]:
    channel.active = False

# Check the status of all channels.
print("Final channel status:")
for channel in fusionln_device.channels:
    print(channel.global_id, '-> active =', "yes" if channel.active else "no")


4. Require TEDS

In [ ]:
# Set TEDS "Usage" to "IfAvailable" for the first 4 channels.
# IMPORTANT: Use "IfAvailable" instead of "Require"!!!
# "TEDS" is a nested property on each channel, and its "Usage" is a selection property
# (Ignore / IfAvailable / Require / Simulation). We set it directly by its label.

# There is a bug: setting TEDS.Usage to "IfAvailable" does not have an effect.
# So we first set it to "Require" and then to "IfAvailable" to work around the bug.
for channel in fusionln_device.channels[:4]:
    teds = channel.get_property_value('TEDS')
    teds.set_property_selection_value('Usage', 'Require')       # work around the bug
    teds.set_property_selection_value('Usage', 'IfAvailable')   # the value we actually want
    # Read the value back to verify it was set correctly.
    print(channel.global_id, '-> TEDS.Usage =', teds.get_property_selection_value('Usage'))


5. Set up the HPF

In [ ]:
# Set the high-pass filter per channel.
# "AnalogPreProcessing" is a nested property on each channel, and its "HpFilter" is a
# selection property. We simply pass the exact filter label we want for each channel.
hpf_per_channel = {
    0: '22.4Hz (0.1dB)',
    1: '0.7Hz (0.1dB)',
    2: 'DC',
    3: '0.7Hz (0.1dB)',
}

for channel_index, hpf_label in hpf_per_channel.items():
    analog_pp = fusionln_device.channels[channel_index].get_property_value('AnalogPreProcessing')
    analog_pp.set_property_selection_value('HpFilter', hpf_label)
    print(f'Channel {channel_index} -> HpFilter = {analog_pp.get_property_selection_value("HpFilter")}')


6. Set the channel input range to 10 V

In [ ]:
# Set the input range to 10 V for all channels.
# Like "HpFilter" above, "InputRange" is a selection property, so we set it directly by label.
# Note, it requires that you know EXACT label of the selection value, which is "10V", not "10 V".
for channel in fusionln_device.channels[:4]:
    analog_pp = channel.get_property_value('AnalogPreProcessing')
    analog_pp.set_property_selection_value('InputRange', '10V')
    # Read the value back to verify it was set correctly.
    print(channel.global_id, '-> InputRange =', analog_pp.get_property_selection_value('InputRange'))


### Now let us list all signals...
For the FusionLN device, we can handle all signals, where each value signal also has a corresponding time and status signal. To differentiate between all of these signals, we request the global ID.

Note: 
- "Time" is the time stamp of each sample in nanoseconds from the Epoch (1.1.1970 0:00)
- "Value" is the signal value reported by the sensor
- "Status" is its quality status. 

### First we define a few auxilary functions

In [ ]:
def _property2string(property_obj, property_name, decoders=('utf-8', 'cp1252', 'latin-1')):
    try:
        return str(property_obj.get_property_value(property_name))
    except UnicodeDecodeError as first_error:
        raw = getattr(first_error, 'object', None)
        if not isinstance(raw, (bytes, bytearray)):
            raise

        last_error = first_error
        for enc in decoders:
            try:
                return raw.decode(enc)
            except Exception as e:
                last_error = e

        # None of the decoders worked; propagate the last failure.
        raise last_error

def get_unit_from_signal(signal):
    descriptor = getattr(signal, 'descriptor', None)
    if descriptor is None and hasattr(signal, 'get_descriptor'):
        descriptor = signal.get_descriptor()
    elif descriptor is None and hasattr(signal, 'getDescriptor'):
        descriptor = signal.getDescriptor()

    if descriptor is None:
        return None

    unit = getattr(descriptor, 'unit', None)
    if unit is None and hasattr(descriptor, 'get_unit'):
        unit = descriptor.get_unit()
    elif unit is None and hasattr(descriptor, 'getUnit'):
        unit = descriptor.getUnit()

    return unit

def set_unit_on_signal(signal, quantity, symbol, unit_id=-1, name=None):
    """Try to (re)build the signal's data descriptor with a new unit and apply it.

    Signals coming from a remote/client device are typically read-only proxies
    for their descriptor, so applying the new descriptor is expected to fail
    (e.g. raise or silently be ignored) unless the signal is a local
    SignalConfig created on this side.
    """
    descriptor = getattr(signal, 'descriptor', None)
    if descriptor is None:
        raise RuntimeError('Signal has no descriptor to base the new unit on')

    existing_unit = getattr(descriptor, 'unit', None)
    if existing_unit is not None:
        unit_builder = daq.UnitBuilderFromExisting(existing_unit)
    else:
        unit_builder = daq.UnitBuilder()
        unit_builder.id = unit_id
    unit_builder.symbol = symbol
    unit_builder.quantity = quantity
    unit_builder.name = name if name is not None else symbol
    new_unit = unit_builder.build()

    descriptor_builder = daq.DataDescriptorBuilderFromExisting(descriptor)
    descriptor_builder.unit = new_unit
    new_descriptor = descriptor_builder.build()

    # This will fail (e.g. AttributeError/RuntimeError) if the signal does not
    # allow its descriptor to be replaced (e.g. a read-only client-side signal).
    signal.descriptor = new_descriptor
    return new_descriptor

def get_unit_from_input_port(input_port):
    signal = None
    if hasattr(input_port, 'signal'):
        signal = input_port.signal
    elif hasattr(input_port, 'get_signal'):
        signal = input_port.get_signal()
    elif hasattr(input_port, 'getSignal'):
        signal = input_port.getSignal()

    if signal is None:
        raise RuntimeError('Input port has no connected signal')

    return get_unit_from_signal(signal)

def format_unit(unit):
    if unit is None:
        return 'None'
    quantity = getattr(unit, 'quantity', None)
    symbol = getattr(unit, 'symbol', None)
    if quantity is not None or symbol is not None:
        return f'quantity={quantity}, symbol={symbol}'
    return str(unit)


### And now iterate through all channels and signals and output available information.

In [ ]:

# List all available channel properties (useful to see what TEDS populated, e.g. sensor/unit info).
print("------------------ Channels ------------------ ")
for channel in fusionln_device.channels:
    print(f"Channel {channel.global_id}")
    for prop in channel.visible_properties:
        value = channel.get_property_value(prop.name)
        print(' ', prop.name, '=', value)
        if prop.value_type.name == 'ctObject':
            # Nested PropertyObject (e.g. TEDS, SensorDescription) - expand its own properties too.
            for sub_prop in value.visible_properties:
                try:
                    sub_value = _property2string(value, sub_prop.name)
                    print('   ', sub_prop.name, '=', sub_value)
                except Exception as e:
                    print(f"    Skipping sub-property {sub_prop.name}: {e}")

# List all signals, their last value, and their unit (quantity + symbol) taken from the
# signal's descriptor. If TEDS was read correctly, the Value signal's quantity/unit
# should reflect the sensor (e.g. Quantity = Acceleration, Symbol = m/s^2).
print("------------------ Signals ------------------ ")
for signal in fusionln_device.signals_recursive:
    try:
        unit = get_unit_from_signal(signal)
        if unit is not None:
            print(f"Signal {signal.global_id}. Last value: {signal.last_value}. Quantity: {unit.quantity}, Unit: {unit.symbol}")
        else:
            print(f"Signal {signal.global_id}. Last value: {signal.last_value}. No unit info.")
    except Exception as e:
        print(f"Signal {signal.global_id}. Could not read details: {e}")



### Our goal is to attach the measurement unit to the signal, so it can propagate through functions blocks and being correctly displayed.

This is not implemented natively in Fusion LN. The work around: use the standard Scaling Function Block to apply the sensitivity values and correct the unit accordingly

The sensitivity is given as "<magnitude> <unit>", e.g. "0.106757  V/(m/s²)" meaning the sensor outputs that many Volts per unit of the physical quantity. For the given Fusion LN type, the channel signal is measured in Volts. Dividing 1 V by the sensitivity, cancels the Volts and leaves the physical quantity's unit, e.g. V / (V/(m/s²)) = m/s².

We will be using the "pint" package for unit arithmetic: https://pint.readthedocs.io

In [ ]:
# The following code uses the pint library to operate with engineering units. 
# You need to install it first, e.g. using pip:
# pip install -q "pint"
# or directly in a Jupyter notebook:
# %pip install -q "pint"

try:
    import pint
except ImportError:
    raise ImportError("pint is not installed. Please install it using pip\n")

ureg = pint.UnitRegistry()

for channel in fusionln_device.channels:
    try:
        sensor_description = channel.get_property_value('SensorDescription')
        if sensor_description is None:
            print(f"Channel {channel.global_id}: No SensorDescription available")
            continue
        sensitivity = _property2string(sensor_description, 'Sensitivity')
        if not sensitivity:
            print(f"Channel {channel.global_id}: No Sensitivity value available")
            continue

        # Sensitivity is given as "<magnitude> <unit>", e.g. "0.106757  V/(m/s²)",
        # meaning the sensor outputs that many Volts per unit of the physical quantity.
        magnitude_str, unit_str = sensitivity.strip().split(None, 1)
        sensitivity_quantity = ureg.Quantity(float(magnitude_str), unit_str)

        # The channel signal itself is measured in Volts. Dividing 1 V by the sensitivity
        # cancels the Volts and leaves the physical quantity's unit, e.g.
        # V / (V/(m/s²)) = m/s²
        channel_unit = (1 * ureg.volt / sensitivity_quantity).to_reduced_units().units

        print(f"Channel {channel.global_id}: Sensitivity = {sensitivity!r}, Channel unit = {channel_unit:~P}")
    except Exception as e:
        print(f"Channel {channel.global_id}: Could not read Sensitivity: {e}")

## Multi Reader

For reading sample-aligned data, we want to use the Multireader, which reads all signals simultaneously from the device. A Multireader can read all the signals if the devices are PTP synchronized and are using the same sample rate. If this is not the case, different Multireaders can be instantiated, and the openDAQ user will need to handle the data alignment (cf. https://docs-dev.opendaq.com/manual/opendaq/dev/howto_guides/howto_read_aligned_signals.html).

In [ ]:
# Collect the "Value" signal of the first 4 channels; these are the signals we want to
# read together, sample-aligned, with a single MultiReader.
# NOTE: find_component() returns a generic IComponent. We cast it to ISignal so the
# objects can also be used where a signal is required (e.g. input_port.connect()).
signals = []
for channel in fusionln_device.channels[:4]:
    signal = daq.ISignal.cast_from(channel.find_component('Sig/Value'))
    print(f"Channel {channel.global_id} -> using signal {signal.global_id}")
    signals.append(signal)

# A MultiReader reads several signals in lock-step (same sample rate, PTP-synchronized).
# Wrapping it in a TimeMultiReader additionally gives us a timestamp for every sample.
reader = daq.MultiReader(signals)
timed_reader = daq.TimeMultiReader(reader)

# The very first read only delivers the initial "signal descriptor changed" event, not
# real samples. We do a 0-sample read to consume that event so the reader is ready to
# hand out actual data afterwards.
reader.read_with_domain(0)

# Right after that event there is no data buffered yet, so the available count is 0.
print(f"Available samples right after start: {reader.available_count}")

# Let the device stream for one second. At 4096 Sa/s that fills the buffer with ~4096
# samples per signal.
time.sleep(1)

# Read one second of data (SAMPLE_RATE samples). read_with_timestamps returns a pair:
#   rr[0] -> the sample values, one array per signal (rr[0][0] is the first signal, ...)
#   rr[1] -> the matching timestamps, one array per signal
rr = timed_reader.read_with_timestamps(SAMPLE_RATE * 1)

# All signals are read in lock-step, so each one yields the same number of samples.
print("Number of samples read per signal:")
for signal_index in range(len(signals)):
    print(f"  signal {signal_index}: {len(rr[0][signal_index])} samples")

# Release the readers so the device stops streaming into them.
timed_reader = None
reader = None


### Plot the data
Now we want also to plot the signals using MATPLOTLIB.

Note: the signal values are in Volts. Sensor sensitivities are not yet taken into account.

In [ ]:
# The following code uses the matplotlib library to plot the signals.
# You need to install it first, e.g. using pip: 
# pip install -q "matplotlib"
# or directly in a Jupyter notebook:
#%pip install -q "matplotlib"

import matplotlib.pyplot as plt
import numpy as np
from datetime import datetime

plt.rcParams["figure.figsize"] = [10, 3.50]
plt.rcParams["figure.autolayout"] = True

plt.title("Fusion LN Signals, Volts")
color = ["red", "green", "blue", "cyan", "magenta", "yellow"]
i = 0
for signal in rr[0][:]:
    plt.plot(rr[1][0], signal, color=color[i])
    i = i + 1

plt.ylabel("Signal Value, V")
plt.xlabel("Timestamp, s")
plt.show()



## Function blocks

In the above sections, we saw how we can use readers to read signal data and its timestamps for use in external Python applications. However, openDAQ offers a different method of data processing in the form of function blocks. Function blocks are contained with modules that are loaded by openDAQ when we create the Instance. Once loaded, they can be instantiated and added to our root device and used for performing data processing tasks. 

As we're focusing on the client-side usage of the SDK with Python, we won't delve into the openDAQ internals of function blocks. But to offer a short explanation - function blocks are compiled pieces of code that are written using openDAQ components and data processing rules. In function block code we define a set of input ports through which we read data, process that data, and output it within our output signals.

First, let's list the available function blocks

In [ ]:
# Print out the names and IDs of available function blocks
for fb_type in daq_instance.available_function_block_types.values():
    print(f"Name:\t{fb_type.Name},\t\t\tID: {fb_type.Id}")


Let's add the renderer function block by utilizing the `RefFBModuleRenderer` to display the signals.

The use of function blocks and the ability to write your own function blocks is the key differentiator compared to the openAPI. openDAQ enables you to write your own small applications faster, as it is a true SDK.

Note: The renderer will open in a small window perhaps hidden behind the text editor


In [ ]:
# Add a renderer function block to display the data, and connect each of our (raw Value)
# signals to its own input port. The renderer creates a new free input port automatically
# every time the previous one is connected, so we just index them 0, 1, 2, ...
renderer = daq_instance.add_function_block('RefFBModuleRenderer')
for i in range(len(signals)):
    renderer.input_ports[i].connect(signals[i])

# Fix the vertical axis range to [-1, 1] instead of letting it auto-scale.
# NOTE: these are the RAW signals in Volts and the channel input range is 10 V, so parts
# of the waveform may fall outside [-1, 1] and be clipped. Widen this range (or set
# UseCustomMinMaxValue = False to auto-scale) if the plot looks flat or empty.
renderer.set_property_value('UseCustomMinMaxValue', True)
renderer.set_property_value('CustomMinValue', -0.1)
renderer.set_property_value('CustomMaxValue', 0.1)


## Let's connect the Scaling FB to the reader, which will apply the sensitivity values as the scaling factor and correct the engineering unit based on the sensitivity.


In [ ]:
# For each channel we add a "RefFBModuleScaling" function block that converts the raw
# Volt signal into the physical quantity, using the sensitivity read from the sensor.
#
# The Scaling FB applies a linear transform:  output = scale * input + offset
#   - scale  = 1 / sensitivity   (V / (V/unit) = unit), so we divide the Volts back out
#   - offset = 0
#   - OutputUnit = the derived engineering unit (e.g. "m/s2", "Pa")
#
# The scaled output signals are collected in "scaled_signals" so they can be fed into a
# reader / renderer further below.

scaling_fbs = []
scaled_signals = []

# Fallback sensor descriptions for non-TEDS channels, keyed by channel index.
# Used when the device does not report a Sensitivity in its SensorDescription.
fallback_sensor_descriptions = {
    1: {"SensorType": "4507", "NominalSensitivity": "10  mV/(m/s²)"},
    2: {"SensorType": "DemoMotor", "NominalSensitivity": "1  V/V"},
}

for channel_index, channel in enumerate(fusionln_device.channels[:4]):
    # Find this channel's "Value" signal to connect as the scaling input.
    value_signal = next(sig for sig in channel.signals if sig.global_id.endswith('/Sig/Value'))

    scaling_fb = daq_instance.add_function_block('RefFBModuleScaling')

    try:
        sensor_description = channel.get_property_value('SensorDescription')
        sensitivity = _property2string(sensor_description, 'Sensitivity') if sensor_description is not None else ''
    except Exception as e:
        sensitivity = ''
        print(f"Channel {channel.global_id}: Could not read Sensitivity: {e}")

    # If the device did not report a Sensitivity, fall back to the known nominal
    # sensitivity for this channel index (non-TEDS channels).
    if not sensitivity:
        fallback = fallback_sensor_descriptions.get(channel_index)
        if fallback:
            sensitivity = fallback['NominalSensitivity']
            print(f"Channel {channel.global_id}: Using nominal sensitivity {sensitivity!r} for {fallback['SensorType']}")

    if sensitivity:
        # Sensitivity is given as "<magnitude> <unit>", e.g. "0.106757  V/(m/s²)".
        magnitude_str, unit_str = sensitivity.strip().split(None, 1)
        sensitivity_quantity = ureg.Quantity(float(magnitude_str), unit_str)

        # 1 V divided by the sensitivity cancels the Volts and leaves the physical
        # quantity, giving us both the scale factor and the resulting unit.
        scaled_quantity = (1 * ureg.volt / sensitivity_quantity).to_reduced_units()
        scale_factor = scaled_quantity.magnitude
        output_unit = f'{scaled_quantity.units:~P}'  # compact symbol, e.g. "m/s²"

        scaling_fb.set_property_value('scale', scale_factor)
        scaling_fb.set_property_value('offset', 0.0)
        scaling_fb.set_property_value('OutputUnit', output_unit)
        print(f"Channel {channel.global_id}: scale = {scale_factor:g}, OutputUnit = {output_unit}")
    else:
        # No sensitivity available: leave the default 1:1 scaling and inherit the input unit.
        print(f"Channel {channel.global_id}: No sensitivity, using default scale = 1 and inherited unit")

    # Connect the raw Value signal to the scaling input and remember the scaled output.
    scaling_fb.input_ports[0].connect(value_signal)
    scaling_fbs.append(scaling_fb)
    scaled_signals.append(scaling_fb.signals[0])


## Visualize the data from the scaling FB

Note: now the sensor values are correctly scalled and the engineering unit is correct

In [ ]:
# Add a renderer function block and connect the SCALED signals (output of the scaling FBs)
# instead of the raw Volt signals, so the plot shows the data in physical units.
renderer = daq_instance.add_function_block('RefFBModuleRenderer')
for i in range(len(scaled_signals)):
    renderer.input_ports[i].connect(scaled_signals[i])

# Let the renderer auto-scale: the scaled data is in engineering units (e.g. m/s²),
# so the previous fixed [-1, 1] range (meant for raw Volts) no longer applies.
renderer.set_property_value('UseCustomMinMaxValue', False)


## Save data in a file
Using the standard WAV Writer function block

Note: the standard WAV write functin block can only save one channel, therefore the output of this function block is four WAV files

In [ ]:
import numpy as np

# ---------------------------------------------------------------------------
# Save 10 seconds of data to WAV files using the standard openDAQ WAVWriter FB
# ("AudioDeviceModuleWavWriter", which implements the IRecorder interface).
#
# We keep the DOUBLE (Float64) values coming out of the Scaling FB, i.e. the data
# in engineering units. Note: the standard WAVWriter accepts Float64 on its input,
# but the underlying miniaudio encoder stores WAV samples as 32-bit float, so the
# file holds the scaled values at 32-bit-float precision.
#
# IMPORTANT - why we re-publish the data instead of connecting the device signal
# directly: the WAVWriter derives the WAV sample rate from the input domain signal
# as (delta / tickResolution) and only works when the domain uses delta = 1. The
# Fusion LN device signal uses delta = 2^20 with tickResolution = 1/2^32, which makes
# that calculation overflow so the encoder never starts (is_recording stays False and
# an empty/no file is produced). To use the standard WAVWriter we therefore:
#   1) read the scaled (Float64) samples from the Scaling FB, then
#   2) re-publish them through a local signal whose domain is WAV-compatible
#      (Int64 time in seconds, Linear rule delta = 1, tickResolution = 1/SAMPLE_RATE),
#   3) connect that signal to the WAVWriter and record.
# ---------------------------------------------------------------------------

RECORD_SECONDS = 10

# --- 1) Read RECORD_SECONDS of the scaled (Float64) signals, sample-aligned. ---
mr = daq.MultiReader(scaled_signals)
tmr = daq.TimeMultiReader(mr)
mr.read_with_domain(0)                 # consume the initial descriptor event
time.sleep(RECORD_SECONDS + 0.5)       # let the device stream RECORD_SECONDS of data
rr = tmr.read_with_timestamps(SAMPLE_RATE * RECORD_SECONDS)
channel_data = [np.asarray(rr[0][i], dtype=np.float64) for i in range(len(scaled_signals))]
tmr = None
mr = None
print("Samples read per channel:", [len(d) for d in channel_data])

# --- 2) Build a WAV-compatible time (domain) descriptor: Int64 seconds, delta = 1. ---
ctx = daq_instance.context

sec_unit = daq.UnitBuilder()
sec_unit.symbol = "s"
sec_unit.name = "seconds"
sec_unit.quantity = "time"
sec_unit.id = -1
sec_unit = sec_unit.build()

time_desc_builder = daq.DataDescriptorBuilder()
time_desc_builder.sample_type = daq.SampleType.Int64
time_desc_builder.unit = sec_unit
time_desc_builder.tick_resolution = daq.Ratio(1, SAMPLE_RATE)
time_desc_builder.rule = daq.LinearDataRule(1, 0)   # delta = 1, start = 0
time_desc = time_desc_builder.build()

value_desc_builder = daq.DataDescriptorBuilder()
value_desc_builder.sample_type = daq.SampleType.Float64   # keep the Scaling FB doubles
value_desc = value_desc_builder.build()

# Remove any WAVWriter FBs left over from previous runs of this cell.
for existing_fb in list(daq_instance.function_blocks):
    if existing_fb.function_block_type.id == 'AudioDeviceModuleWavWriter':
        daq_instance.remove_function_block(existing_fb)

# --- 3) One WAVWriter (and one .wav file) per channel. ---
wav_files = []
for channel_index, values in enumerate(channel_data):
    n = len(values)
    if n == 0:
        print(f"Channel {channel_index}: no samples, skipping")
        continue

    file_name = f'scaled_channel_{channel_index}.wav'

    # Local signal that carries the scaled data with a WAV-compatible domain.
    time_signal = daq.Signal(ctx, None, f'wavTime{channel_index}', None)
    time_signal.descriptor = time_desc
    value_signal = daq.Signal(ctx, None, f'wavValue{channel_index}', None)
    value_signal.descriptor = value_desc
    value_signal.domain_signal = time_signal

    wav_writer = daq_instance.add_function_block('AudioDeviceModuleWavWriter')
    wav_writer.set_property_value('FileName', file_name)
    wav_writer.input_ports[0].connect(value_signal)

    recorder = daq.IRecorder.cast_from(wav_writer)
    # The writer only learns the input descriptors after processing the initial
    # descriptor event on the scheduler, so retry start_recording until it takes.
    for _ in range(50):
        time.sleep(0.05)
        recorder.start_recording()
        if recorder.is_recording:
            break
    if not recorder.is_recording:
        print(f"Channel {channel_index}: could not start recording, skipping")
        daq_instance.remove_function_block(wav_writer)
        continue

    # Send the RECORD_SECONDS of scaled samples as one packet.
    domain_packet = daq.DataPacket(time_desc, n, 0)
    data_packet = daq.DataPacketWithDomain(domain_packet, value_desc, n, 0)
    np.frombuffer(data_packet.data, dtype=np.float64)[:] = values
    value_signal.send_packet(data_packet)
    time.sleep(0.5)   # let the writer encode the packet

    recorder.stop_recording()
    daq_instance.remove_function_block(wav_writer)
    wav_files.append(file_name)
    print(f"Channel {channel_index}: wrote {n} samples -> {file_name}")

# --- Report the resulting files. ---
print("\nWAV files written:")
for file_name in wav_files:
    abs_path = os.path.abspath(file_name)
    size_kb = os.path.getsize(abs_path) / 1024 if os.path.exists(abs_path) else 0
    print(f'  "{abs_path}" ({size_kb:.1f} KiB)')


### Let's validate the files.

Read and compare with the original data

In [ ]:
# Validate the saved WAV files against the original data in "channel_data".
#
# We read each .wav back (they are mono, 32-bit IEEE float, fs = SAMPLE_RATE) and
# overlay 1 second of the WAV samples on top of the original scaled samples.
# One subplot per channel.
import matplotlib.pyplot as plt
from scipy.io import wavfile

PLOT_SECONDS = 1
n_plot = SAMPLE_RATE * PLOT_SECONDS

n_channels = len(wav_files)
fig, axes = plt.subplots(n_channels, 1, figsize=(11, 2.4 * n_channels),
                         sharex=True, squeeze=False)
axes = axes[:, 0]

for channel_index, (ax, file_name) in enumerate(zip(axes, wav_files)):
    fs, wav_values = wavfile.read(file_name)
    wav_values = np.asarray(wav_values, dtype=np.float64)

    original = np.asarray(channel_data[channel_index], dtype=np.float64)
    n = min(n_plot, len(wav_values), len(original))
    t = np.arange(n) / SAMPLE_RATE

    # Numeric check over the full record (WAV holds 32-bit floats, so compare
    # against the original cast to float32 precision).
    n_all = min(len(wav_values), len(original))
    max_diff = np.max(np.abs(wav_values[:n_all] - original[:n_all]))
    print(f'{file_name}: fs={fs} Hz, {len(wav_values)} samples, '
          f'max |WAV - original| = {max_diff:.3e}')

    # Engineering unit for the Y-axis label.
    try:
        unit_lab = scaled_signals[channel_index].descriptor.unit.symbol or 'EU'
    except Exception:
        unit_lab = 'EU'

    ax.plot(t, original[:n], linewidth=1.4)          # original channel_data
    ax.plot(t, wav_values[:n], linewidth=0.8)        # data read back from the WAV
    ax.set_title(f'Channel {channel_index} - {file_name}', loc='left', fontsize=9)
    ax.set_ylabel(f'Amplitude [{unit_lab}]')
    ax.grid(False)

axes[-1].set_xlabel('Time [s]')
fig.suptitle(f'WAV files vs. original data - first {PLOT_SECONDS} s')
fig.tight_layout()
plt.show()


### Now save as a binary UFF

Note: it uses pyUFF for saving the data

In [ ]:
# Save the 10 seconds of data into a SINGLE binary UFF file, dataset Type 58b.

# We will be using the pyuff library to write the data into a UFF file. 
# You need to install it first, e.g. using pip:
# %pip install pyuff
# or directly in a Jupyter notebook:
# %pip install pyuff

#
# We reuse "channel_data" (the scaled Float64 values in engineering units, read in the
# previous cell). Each channel becomes one dataset 58 (Time Response); passing binary=1
# writes it in the BINARY variant (58b). All datasets go into one .uff file.
#
# NOTE on pyuff (2.5.x): the binary writer truncates the file when a set is written in
# 'overwrite' mode, which corrupts the first dataset. We therefore start from a fresh
# (deleted) file and write every dataset in 'add' mode, which appends correctly.

try:
    import pyuff
except ImportError:
    raise ImportError("pyuff is not installed. Please install it.\n")

# The file name for the UFF file to be created. You can change it to your preferred name.
uff_filename = 'scaled_data_type58b.uff'

datasets = []
for channel_index, values in enumerate(channel_data):
    n = len(values)
    if n == 0:
        print(f"Channel {channel_index}: no samples, skipping")
        continue

    # Ordinate unit label from the scaled signal's descriptor (e.g. "Pa", "m/s^2").
    try:
        unit_lab = scaled_signals[channel_index].descriptor.unit.symbol or 'EU'
    except Exception:
        unit_lab = 'EU'

    time_axis = np.arange(n) / SAMPLE_RATE   # abscissa in seconds

    ds = pyuff.prepare_58(
        binary=1,                              # 1 => dataset 58 in BINARY form (58b)
        func_type=1,                           # 1 = Time Response
        ver_num=1,
        load_case_id=0,
        rsp_ent_name='NONE', rsp_node=channel_index + 1, rsp_dir=1,  # response DOF
        ref_ent_name='NONE', ref_node=1, ref_dir=1,                  # reference DOF
        data=values.astype(np.float64),        # ordinate (double precision)
        x=time_axis,                           # abscissa (time, s)
        ord_data_type=4,                       # 4 = real, double precision
        abscissa_spacing=1,                    # 1 = even spacing
        abscissa_min=0.0,
        abscissa_inc=1.0 / SAMPLE_RATE,        # dt = 1 / fs
        z_axis_value=0.0,
        abscissa_spec_data_type=17,            # 17 = time
        abscissa_len_unit_exp=0, abscissa_force_unit_exp=0, abscissa_temp_unit_exp=0,
        ordinate_spec_data_type=0,
        ordinate_len_unit_exp=0, ordinate_force_unit_exp=0, ordinate_temp_unit_exp=0,
        orddenom_spec_data_type=0,
        orddenom_len_unit_exp=0, orddenom_force_unit_exp=0, orddenom_temp_unit_exp=0,
        z_axis_spec_data_type=0,
        z_axis_len_unit_exp=0, z_axis_force_unit_exp=0, z_axis_temp_unit_exp=0,
        abscissa_axis_units_lab='s',
        ordinate_axis_units_lab=unit_lab,
        id1=f'Fusion LN channel {channel_index} - time response',
        return_full_dict=True,
    )
    ds['func_id'] = 0   # function identification number (not exposed by prepare_58)
    datasets.append(ds)

# Write all channel datasets into a single binary UFF file (append mode, fresh file).
if os.path.exists(uff_filename):
    os.remove(uff_filename)
uff = pyuff.UFF(uff_filename)
uff.write_sets(datasets, mode='add')

abs_uff = os.path.abspath(uff_filename)
print(f"Wrote {len(datasets)} Type 58b dataset(s) to")
print(f'  "{abs_uff}" ({os.path.getsize(abs_uff) / 1024:.1f} KiB)')

# Quick read-back verification.
check = pyuff.UFF(uff_filename)
print("Dataset types in file:", list(check.get_set_types()))
for i, s in enumerate(check.read_sets()):
    print(f"  dataset {i}: binary={s.get('binary')}, points={len(s['data'])}, "
          f"dt={s.get('abscissa_inc'):.6g}s, unit={s.get('ordinate_axis_units_lab')!r}, "
          f"id1={s.get('id1')!r}")


## Check: read the UFF file and plot its content using MATPLOTLIB

In [ ]:
# Read back the binary UFF (Type 58b) file we just wrote and compare it with the
# original data in "channel_data": both traces are plotted on top of each other,
# one subplot per channel.
import matplotlib.pyplot as plt
import pyuff

PLOT_SECONDS = 1
n_plot = SAMPLE_RATE * PLOT_SECONDS

# Open the UFF file and read all datasets.
reader = pyuff.UFF(uff_filename)
sets = reader.read_sets()
if isinstance(sets, dict):        # a single dataset is returned as a plain dict
    sets = [sets]

# Keep only the Type 58 (function / time-response) datasets.
sets = [s for s in sets if s.get('type') == 58]
print(f"Read {len(sets)} Type 58 dataset(s) from "
      f'"{os.path.abspath(uff_filename)}"')

# One stacked subplot per channel, sharing the time axis.
fig, axes = plt.subplots(len(sets), 1, figsize=(11, 2.4 * len(sets)),
                         sharex=True, squeeze=False)
axes = axes[:, 0]

for channel_index, (ax, s) in enumerate(zip(axes, sets)):
    y = np.asarray(s['data'], dtype=np.float64)
    # Rebuild the time axis: evenly spaced -> use abscissa_min + i*abscissa_inc.
    if s.get('abscissa_spacing', 1) == 1:
        dt = s.get('abscissa_inc') or (1.0 / SAMPLE_RATE)
        t_full = s.get('abscissa_min', 0.0) + np.arange(len(y)) * dt
    else:
        t_full = np.asarray(s['x'], dtype=np.float64)

    original = np.asarray(channel_data[channel_index], dtype=np.float64)

    # Numeric check over the full record (UFF 58b stores doubles -> exact match).
    n_all = min(len(y), len(original))
    max_diff = np.max(np.abs(y[:n_all] - original[:n_all]))
    print(f'  channel {channel_index}: {len(y)} samples, '
          f'max |UFF - original| = {max_diff:.3e}')

    n = min(n_plot, len(y), len(original))
    unit = s.get('ordinate_axis_units_lab') or 'EU'
    ax.plot(t_full[:n], original[:n], linewidth=1.4)   # original channel_data
    ax.plot(t_full[:n], y[:n], linewidth=0.8)          # data read back from the UFF
    ax.set_ylabel(f'Amplitude [{unit}]')
    ax.set_title(s.get('id1') or f'channel {channel_index}', loc='left', fontsize=9)
    ax.grid(False)

axes[-1].set_xlabel('Time [s]')
fig.suptitle(f'{os.path.basename(uff_filename)} vs. original data - first {PLOT_SECONDS} s')
fig.tight_layout()
plt.show()


### Now let's use standard Python tools to save the same 10 seconds of data as a CSV file.

Note: the first column is the time stamp, the next four columns are the values. 
Note: the very first row is the header row, must be skipped when improting the file.

In [ ]:
# Save the same 10 seconds of data as a CSV file using standard Python tools.
#
# Layout:
#   column 1      -> time stamp [s]
#   columns 2..5  -> the four channel values (engineering units)
#   row 1         -> header row (to be skipped when the file is imported later)
import csv

csv_filename = 'scaled_data.csv'

n_samples = min(len(ch) for ch in channel_data)     # all channels have the same length
time_stamps = np.arange(n_samples) / SAMPLE_RATE    # 0, 1/fs, 2/fs, ...

# Header: time column + one column per channel (with its engineering unit).
header = ['time_s']
for channel_index in range(len(channel_data)):
    header.append(f'channel_{channel_index}')

with open(csv_filename, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(header)                          # first (header) row
    for i in range(n_samples):
        writer.writerow([time_stamps[i]] + [channel_data[ch][i] for ch in range(len(channel_data))])

abs_csv = os.path.abspath(csv_filename)
print(f'Wrote {n_samples} rows x {len(header)} columns to')
print(f'  "{abs_csv}" ({os.path.getsize(abs_csv) / 1024:.1f} KiB)')
print('Header:', ','.join(header))

# Import check: skip the header row (skiprows=1) and confirm the shape.
imported = np.loadtxt(csv_filename, delimiter=',', skiprows=1)
print(f'Re-imported (header skipped): {imported.shape[0]} rows x {imported.shape[1]} columns')
print('First data row:', imported[0])


### Validate the CSV file as well

In [ ]:
# Read back the CSV file and compare it with the original data in "channel_data":
# both traces are plotted on top of each other, one subplot per channel.
import matplotlib.pyplot as plt

PLOT_SECONDS = 1
n_plot = SAMPLE_RATE * PLOT_SECONDS

# Import the CSV, skipping the header row.
imported = np.loadtxt(csv_filename, delimiter=',', skiprows=1)
csv_time = imported[:, 0]            # first column  -> time stamps
csv_values = imported[:, 1:]         # remaining columns -> channel values
print(f'Read {imported.shape[0]} rows x {imported.shape[1]} columns from '
      f'"{os.path.abspath(csv_filename)}"')

n_channels = csv_values.shape[1]
fig, axes = plt.subplots(n_channels, 1, figsize=(11, 2.4 * n_channels),
                         sharex=True, squeeze=False)
axes = axes[:, 0]

for channel_index, ax in enumerate(axes):
    y = csv_values[:, channel_index]
    original = np.asarray(channel_data[channel_index], dtype=np.float64)

    # Numeric check over the full record.
    n_all = min(len(y), len(original))
    max_diff = np.max(np.abs(y[:n_all] - original[:n_all]))
    print(f'  channel {channel_index}: {len(y)} samples, '
          f'max |CSV - original| = {max_diff:.3e}')

    n = min(n_plot, len(y), len(original))
    try:
        unit_lab = scaled_signals[channel_index].descriptor.unit.symbol or 'EU'
    except Exception:
        unit_lab = 'EU'

    ax.plot(csv_time[:n], original[:n], linewidth=1.4)   # original channel_data
    ax.plot(csv_time[:n], y[:n], linewidth=0.8)          # data read back from the CSV
    ax.set_ylabel(f'Amplitude [{unit_lab}]')
    ax.set_title(f'Channel {channel_index}', loc='left', fontsize=9)
    ax.grid(False)

axes[-1].set_xlabel('Time [s]')
fig.suptitle(f'{os.path.basename(csv_filename)} vs. original data - first {PLOT_SECONDS} s')
fig.tight_layout()
plt.show()
